# Proyecto 2 - Segmentación de Clientes
### Técnicas de Aprendizaje de Máquina
### Pontificia Universidad Javeriana

**Objetivo:** Aplicar técnicas de clustering no supervisado (K-Means) para segmentar clientes, y luego utilizar esos segmentos como etiquetas para entrenar un modelo supervisado de árbol de decisión que permita predecir el segmento de un cliente nuevo.

---

## Descripción del Problema

La segmentación de clientes es una estrategia clave en el mundo empresarial que permite dividir la base de clientes en grupos con comportamientos y características similares. Esto facilita la creación de estrategias personalizadas de marketing, mejoras en productos y servicios, y una mejor comprensión del comportamiento del consumidor.

En este proyecto se trabaja con un dataset de clientes que contiene variables demográficas, de comportamiento de compra y de satisfacción. El flujo de trabajo será:
1. Explorar y preprocesar los datos.
2. Aplicar K-Means para identificar segmentos naturales.
3. Usar los clusters como etiquetas para entrenar un árbol de decisión.
4. Interpretar los resultados y extraer conclusiones de negocio.

---
## 1. Análisis Exploratorio y Preprocesamiento de Datos (25%)

### 1.1 Importación de Librerías

In [ ]:
# === Librerías Estándar ===
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# === Visualización ===
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

# === Scikit-learn: Preprocesamiento ===
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.decomposition import PCA

# === Scikit-learn: Clustering ===
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# === Scikit-learn: Árbol de Decisión ===
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, ConfusionMatrixDisplay
)

print('Todas las librerías cargadas correctamente.')

### 1.2 Carga del Dataset

In [ ]:
df = pd.read_csv('dataset_proyecto_2.csv')

print(f'Shape del dataset: {df.shape}')
print(f'Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}')
df.head(10)

### 1.3 Descripción del Dataset

El dataset contiene **13 variables** que describen el comportamiento de compra y características de los clientes:

| Variable | Tipo | Descripción |
|---|---|---|
| `edad` | Numérica entera | Edad del cliente en años |
| `ingresos_anuales` | Numérica continua | Ingreso anual estimado (USD) |
| `cantidad_compras` | Numérica entera | Número total de compras realizadas |
| `valor_promedio_compra` | Numérica continua | Valor promedio por compra (USD) |
| `frecuencia_compras_mensual` | Numérica continua | Promedio de compras por mes |
| `dispositivo_utilizado` | Categórica | Dispositivo más usado (móvil, PC, tablet) |
| `fuente_trafico` | Categórica | Canal de adquisición |
| `dias_desde_ultima_compra` | Numérica entera | Días desde la última compra |
| `valor_total_gastado` | Numérica continua | Total gastado en todas las compras (USD) |
| `satisfaccion_cliente` | Numérica continua | Nivel de satisfacción (1 a 5) |
| `metodo_pago` | Categórica | Método de pago más frecuente |
| `participacion_programa_lealtad` | Binaria (0/1) | Participa en programa de lealtad |
| `productos_adquiridos` | Categórica | Categoría principal de productos |

In [ ]:
print('=== Información general del dataset ===')
df.info()
print()
print('=== Tipos de variables ===')
print(df.dtypes)

In [ ]:
print('=== Estadísticas Descriptivas ===')
df.describe().round(2)

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Variables categóricas: {cat_cols}\n')

for col in cat_cols:
    print(f'--- {col} ---')
    print(df[col].value_counts())
    print()

### 1.4 Valores Nulos y Duplicados

In [ ]:
nulos = df.isnull().sum()
pct_nulos = (nulos / len(df) * 100).round(2)

resumen_nulos = pd.DataFrame({'Valores nulos': nulos, 'Porcentaje (%)': pct_nulos})
print('=== Valores Nulos por Columna ===')
if nulos.sum() > 0:
    print(resumen_nulos[resumen_nulos['Valores nulos'] > 0])
else:
    print('No hay valores nulos.')

n_dup = df.duplicated().sum()
print(f'\nRegistros duplicados: {n_dup}')

In [ ]:
# Manejo de valores nulos
# Variables numéricas -> mediana (robusta ante outliers)
# Variables categóricas -> moda

num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

for col in num_cols:
    if df[col].isnull().sum() > 0:
        mediana = df[col].median()
        df[col].fillna(mediana, inplace=True)
        print(f'[NUM] {col}: imputado con mediana = {mediana:.2f}')

for col in cat_cols:
    if df[col].isnull().sum() > 0:
        moda = df[col].mode()[0]
        df[col].fillna(moda, inplace=True)
        print(f'[CAT] {col}: imputado con moda = {moda}')

if n_dup > 0:
    df.drop_duplicates(inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f'Se eliminaron {n_dup} registros duplicados.')

print('Limpieza completada. Shape final:', df.shape)

### 1.5 Visualización y Análisis Exploratorio

In [ ]:
num_features = ['edad', 'ingresos_anuales', 'cantidad_compras',
                'valor_promedio_compra', 'frecuencia_compras_mensual',
                'dias_desde_ultima_compra', 'valor_total_gastado', 'satisfaccion_cliente']

fig, axes = plt.subplots(3, 3, figsize=(15, 11))
axes = axes.flatten()

for i, col in enumerate(num_features):
    axes[i].hist(df[col], bins=30, color=sns.color_palette('Set2')[i % 8], edgecolor='white')
    axes[i].set_title(f'Distribución: {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frecuencia')

axes[-1].set_visible(False)
plt.suptitle('Distribución de Variables Numéricas', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    counts = df[col].value_counts()
    axes[i].bar(counts.index, counts.values, color=sns.color_palette('Set2', len(counts)))
    axes[i].set_title(f'Distribución: {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Conteo')
    axes[i].tick_params(axis='x', rotation=20)

plt.suptitle('Distribución de Variables Categóricas', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
corr_matrix = df[num_features + ['participacion_programa_lealtad']].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, square=True,
    linewidths=0.5, ax=ax
)
ax.set_title('Matriz de Correlación entre Variables Numéricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Observaciones:')
print('- Se espera alta correlación entre valor_total_gastado y cantidad_compras.')
print('- La frecuencia de compras y valor promedio pueden estar inversamente relacionados.')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(num_features):
    axes[i].boxplot(df[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor=sns.color_palette('Set2')[i % 8], alpha=0.8))
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xticks([])

plt.suptitle('Boxplots - Detección de Outliers', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 1.6 Identificación y Tratamiento de Outliers

Se utiliza el método IQR (Rango Intercuartílico) para detectar valores atípicos. Para clustering, los outliers extremos pueden distorsionar los centroides. Se aplica **winsorization (capping)** al rango IQR: se reemplazan los extremos por los límites inferior y superior, preservando la cantidad total de filas.

> **Justificación:** Eliminar filas con outliers reduciría el tamaño del dataset y podría perder clientes reales (e.g., clientes VIP con gasto muy alto). La winsorization es más conservadora y adecuada para segmentación.

In [ ]:
def detectar_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR
    n = ((series < lim_inf) | (series > lim_sup)).sum()
    return n, lim_inf, lim_sup

print('=== Detección de Outliers (método IQR) ===')
outliers_info = {}
for col in num_features:
    n, lim_inf, lim_sup = detectar_outliers_iqr(df[col])
    outliers_info[col] = (n, lim_inf, lim_sup)
    if n > 0:
        print(f'{col}: {n} outliers | Rango válido: [{lim_inf:.2f}, {lim_sup:.2f}]')

df_clean = df.copy()
print('\n=== Aplicando Winsorization ===')
for col in num_features:
    n, lim_inf, lim_sup = outliers_info[col]
    if n > 0:
        df_clean[col] = df_clean[col].clip(lower=lim_inf, upper=lim_sup)
        print(f'  {col}: recortado a [{lim_inf:.2f}, {lim_sup:.2f}]')

print('\nOutliers tratados. Shape:', df_clean.shape)

### 1.7 Codificación de Variables Categóricas

Se aplica **Label Encoding** a las variables categóricas. Esto transforma cada categoría en un número entero, lo que permite incluirlas en el proceso de K-Means y en el árbol de decisión.

> **Justificación:** LabelEncoding es adecuado aquí porque K-Means necesita valores numéricos para el cálculo de distancias. Para el árbol de decisión, los splits binarios en cada nodo funcionan correctamente con valores enteros codificados.

In [ ]:
df_encoded = df_clean.copy()
encoders = {}
label_maps = {}

for col in cat_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
    encoders[col] = le
    label_maps[col] = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f'{col}: {label_maps[col]}')

print('\nCodificación completada.')
df_encoded.head()

### 1.8 Normalización / Estandarización de Variables Numéricas

K-Means es sensible a la escala de las variables porque usa distancias euclidianas. Se aplica **StandardScaler** (Z-score: media=0, desviación estándar=1) a todas las variables antes del clustering.

> **Justificación:** Sin escalado, variables como `ingresos_anuales` (en miles de USD) dominarían completamente sobre `satisfaccion_cliente` (escala 1-5), haciendo que los clusters reflejen diferencias de magnitud y no de comportamiento real.

In [ ]:
all_features = list(dict.fromkeys(
    num_features + cat_cols + ['participacion_programa_lealtad']
))

X = df_encoded[all_features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=all_features)

print(f'Forma de X_scaled: {X_scaled.shape}')
print('\nEstadísticas post-escalado (media ≈ 0, std ≈ 1):')
print(X_scaled_df.describe().round(3).loc[['mean', 'std']])

---
## 2. Aplicación de K-Means (25%)

### 2.1 Determinación del Número Óptimo de Clusters

Se combina el **Método del Codo** (busca el punto de inflexión en la inercia) con el **Coeficiente de Silueta** (mide qué tan bien separados están los clusters). Se evaluarán valores de k entre 2 y 10.

In [ ]:
rango_k = range(2, 11)
inercias = []
silhouette_scores = []

for k in rango_k:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_scaled)
    inercias.append(km.inertia_)
    sil = silhouette_score(X_scaled, km.labels_, sample_size=min(5000, len(X_scaled)))
    silhouette_scores.append(sil)
    print(f'k={k:2d} | Inercia: {km.inertia_:10.2f} | Silhouette: {sil:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Método del Codo
axes[0].plot(list(rango_k), inercias, 'o-', color='steelblue', linewidth=2.5, markersize=8)
axes[0].set_xlabel('Número de Clusters (k)', fontsize=12)
axes[0].set_ylabel('Inercia (WCSS)', fontsize=12)
axes[0].set_title('Método del Codo', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.4)
axes[0].xaxis.set_major_locator(mticker.MultipleLocator(1))

# Silhouette Score
axes[1].plot(list(rango_k), silhouette_scores, 's-', color='coral', linewidth=2.5, markersize=8)
axes[1].set_xlabel('Número de Clusters (k)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Coeficiente de Silueta', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.4)
axes[1].xaxis.set_major_locator(mticker.MultipleLocator(1))

plt.suptitle('Determinación del Número Óptimo de Clusters', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

best_k = list(rango_k)[np.argmax(silhouette_scores)]
print(f'Mejor k según Silhouette Score: k = {best_k} (score = {max(silhouette_scores):.4f})')
print('Se recomienda revisar visualmente ambas gráficas para confirmar la elección.')

In [ ]:
# K seleccionado - se puede modificar manualmente si el análisis visual lo sugiere
K_OPTIMO = best_k
print(f'K seleccionado: {K_OPTIMO}')

### 2.2 Aplicación de K-Means con k óptimo

Se usa la inicialización **k-means++** que selecciona centroides iniciales de forma inteligente, reduciendo la probabilidad de convergencia a mínimos locales. Se ejecutan 15 inicializaciones diferentes y se conserva la mejor.

In [ ]:
kmeans_final = KMeans(
    n_clusters=K_OPTIMO,
    init='k-means++',
    n_init=15,
    max_iter=300,
    random_state=42
)
kmeans_final.fit(X_scaled)

df_clean['cluster'] = kmeans_final.labels_
df_encoded['cluster'] = kmeans_final.labels_

dist = df_clean['cluster'].value_counts().sort_index()
print(f'Inercia final:       {kmeans_final.inertia_:.2f}')
print(f'Silhouette Score:    {silhouette_score(X_scaled, kmeans_final.labels_):.4f}')
print()
print('Distribución de clientes por cluster:')
for k, n in dist.items():
    print(f'  Cluster {k}: {n:,} clientes ({n/len(df_clean)*100:.1f}%)')

In [ ]:
# Visualización de clusters con PCA (reducción a 2D para graficación)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
pca_df['cluster'] = kmeans_final.labels_

fig, ax = plt.subplots(figsize=(10, 7))
palette = sns.color_palette('Set1', K_OPTIMO)

for k in range(K_OPTIMO):
    mask = pca_df['cluster'] == k
    ax.scatter(
        pca_df.loc[mask, 'PC1'], pca_df.loc[mask, 'PC2'],
        color=palette[k], label=f'Cluster {k}',
        alpha=0.6, s=40, edgecolors='white', linewidths=0.3
    )

centroids_pca = pca.transform(kmeans_final.cluster_centers_)
ax.scatter(
    centroids_pca[:, 0], centroids_pca[:, 1],
    c='black', marker='X', s=200, zorder=5, label='Centroides'
)

ax.set_title(f'Visualización de Clusters K-Means (k={K_OPTIMO}) — PCA 2D',
             fontsize=13, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% varianza)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% varianza)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Varianza explicada por 2 componentes PCA: {sum(pca.explained_variance_ratio_)*100:.1f}%')

### 2.3 Descripción Detallada de los Segmentos de Clientes

In [ ]:
perfil_clusters = df_clean.groupby('cluster')[
    num_features + ['participacion_programa_lealtad']
].mean().round(2)

print('=== Perfil Promedio por Cluster ===')
perfil_clusters

In [ ]:
# Heatmap de perfiles normalizados con valores reales anotados
mms = MinMaxScaler()
perfil_norm = pd.DataFrame(
    mms.fit_transform(perfil_clusters),
    columns=perfil_clusters.columns,
    index=perfil_clusters.index
)

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(
    perfil_norm, annot=perfil_clusters.values, fmt='.1f',
    cmap='YlOrRd', linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Valor Normalizado (0-1)'}
)
ax.set_title(
    'Perfil Normalizado de Cada Cluster (valores anotados = promedios reales)',
    fontsize=13, fontweight='bold'
)
ax.set_xlabel('Variable')
ax.set_ylabel('Cluster')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots comparativos por cluster
key_vars = ['ingresos_anuales', 'valor_total_gastado',
            'frecuencia_compras_mensual', 'satisfaccion_cliente',
            'dias_desde_ultima_compra', 'edad']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, var in enumerate(key_vars):
    data_box = [df_clean.loc[df_clean['cluster'] == k, var].values for k in range(K_OPTIMO)]
    bp = axes[i].boxplot(data_box, patch_artist=True,
                          labels=[f'C{k}' for k in range(K_OPTIMO)])
    for patch, color in zip(bp['boxes'], sns.color_palette('Set1', K_OPTIMO)):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    axes[i].set_title(var, fontweight='bold')
    axes[i].set_xlabel('Cluster')
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Distribución de Variables Clave por Cluster', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Variables categóricas por cluster
print('=== Variables Categóricas por Cluster (% por fila) ===')
for col in cat_cols:
    print(f'\n--- {col} ---')
    tabla = pd.crosstab(
        df_clean['cluster'], df_clean[col], normalize='index'
    ).round(3) * 100
    print(tabla.to_string())
    print('(%)')

In [ ]:
# Descripción textual de cada segmento
print('=' * 60)
print('DESCRIPCIÓN DE SEGMENTOS DE CLIENTES')
print('=' * 60)

for k in range(K_OPTIMO):
    subset = df_clean[df_clean['cluster'] == k]
    print(f'\nCLUSTER {k} — {len(subset):,} clientes ({len(subset)/len(df_clean)*100:.1f}%)')
    print(f'  Edad promedio:            {subset["edad"].mean():.1f} años')
    print(f'  Ingresos anuales prom.:   ${subset["ingresos_anuales"].mean():,.0f}')
    print(f'  Total gastado prom.:      ${subset["valor_total_gastado"].mean():,.0f}')
    print(f'  Compras/mes (prom.):      {subset["frecuencia_compras_mensual"].mean():.2f}')
    print(f'  Satisfacción prom.:       {subset["satisfaccion_cliente"].mean():.2f}/5')
    print(f'  Días últ. compra (prom.): {subset["dias_desde_ultima_compra"].mean():.1f}')
    print(f'  % En programa lealtad:   {subset["participacion_programa_lealtad"].mean()*100:.1f}%')
    print(f'  Dispositivo más común:   {subset["dispositivo_utilizado"].mode()[0]}')
    print(f'  Fuente tráfico ppal.:    {subset["fuente_trafico"].mode()[0]}')
    print(f'  Método pago más común:   {subset["metodo_pago"].mode()[0]}')
    print(f'  Producto ppal.:          {subset["productos_adquiridos"].mode()[0]}')

---
## 3. Entrenamiento y Evaluación del Árbol de Decisión (20%)

### 3.1 Preparación de Datos

In [ ]:
X_clf = df_encoded[all_features].copy()
y_clf = df_encoded['cluster'].copy()

# División estratificada: mantiene la proporción de clases en train y test
X_train, X_test, y_train, y_test = train_test_split(
    X_clf, y_clf,
    test_size=0.25,
    random_state=42,
    stratify=y_clf
)

print(f'Dataset total:       {len(X_clf):,} muestras')
print(f'Entrenamiento:       {len(X_train):,} ({len(X_train)/len(X_clf)*100:.1f}%)')
print(f'Prueba:              {len(X_test):,} ({len(X_test)/len(X_clf)*100:.1f}%)')
print()
print('Distribución de clases en train:')
print(y_train.value_counts().sort_index())
print('\nDistribución de clases en test:')
print(y_test.value_counts().sort_index())

### 3.2 Búsqueda de Profundidad Óptima

In [ ]:
profundidades = range(2, 15)
acc_train_list = []
acc_test_list = []

for d in profundidades:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42, criterion='gini')
    dt.fit(X_train, y_train)
    acc_train_list.append(dt.score(X_train, y_train))
    acc_test_list.append(dt.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(list(profundidades), acc_train_list, 'o-', label='Train', color='steelblue', linewidth=2)
ax.plot(list(profundidades), acc_test_list, 's-', label='Test', color='coral', linewidth=2)
ax.set_xlabel('Profundidad Máxima (max_depth)', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Accuracy vs Profundidad del Árbol', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
plt.tight_layout()
plt.show()

best_depth = list(profundidades)[np.argmax(acc_test_list)]
print(f'Profundidad óptima según accuracy en test: {best_depth}')
print(f'Accuracy en test: {max(acc_test_list):.4f}')

### 3.3 Entrenamiento del Modelo Final

In [ ]:
MAX_DEPTH = best_depth

dt_final = DecisionTreeClassifier(
    max_depth=MAX_DEPTH,
    criterion='gini',
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)
dt_final.fit(X_train, y_train)

y_pred = dt_final.predict(X_test)
y_pred_train = dt_final.predict(X_train)

print('=== Modelo entrenado exitosamente ===')
print(f'Accuracy Train: {accuracy_score(y_train, y_pred_train):.4f}')
print(f'Accuracy Test:  {accuracy_score(y_test, y_pred):.4f}')

In [ ]:
# Visualización del árbol (primeros 4 niveles para legibilidad)
fig, ax = plt.subplots(figsize=(22, 10))
plot_tree(
    dt_final,
    feature_names=all_features,
    class_names=[f'Cluster {k}' for k in range(K_OPTIMO)],
    filled=True,
    rounded=True,
    max_depth=min(4, MAX_DEPTH),
    fontsize=8,
    ax=ax,
    impurity=False
)
ax.set_title(
    f'Árbol de Decisión — Primeros {min(4, MAX_DEPTH)} niveles',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.show()

### 3.4 Importancia de Variables

In [ ]:
importancias = pd.Series(dt_final.feature_importances_, index=all_features)
importancias = importancias.sort_values(ascending=False)

print('=== Importancia de Variables (Gini Importance) ===')
for feat, imp in importancias.items():
    barra = '|' * int(imp * 50)
    print(f'  {feat:35s} {imp:.4f}  {barra}')

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#e74c3c' if i < 3 else '#3498db' for i in range(len(importancias))]
bars = ax.barh(
    importancias.index[::-1], importancias.values[::-1],
    color=colors[::-1], edgecolor='white'
)
for bar, val in zip(bars, importancias.values[::-1]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

ax.set_xlabel('Importancia (Gini)', fontsize=12)
ax.set_title('Importancia de Variables — Árbol de Decisión', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print(f'\nTop 3 variables: {importancias.head(3).index.tolist()}')

### 3.5 Evaluación del Modelo

In [ ]:
print('=== Reporte de Clasificación (Conjunto de Prueba) ===')
print(classification_report(
    y_test, y_pred,
    target_names=[f'Cluster {k}' for k in range(K_OPTIMO)]
))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[f'C{k}' for k in range(K_OPTIMO)]
)
disp.plot(ax=ax, cmap='Blues', colorbar=True)
ax.set_title('Matriz de Confusión — Árbol de Decisión', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Validación cruzada 5-fold
cv_scores = cross_val_score(dt_final, X_clf, y_clf, cv=5, scoring='accuracy')

print('=== Validación Cruzada (5-Fold) ===')
print(f'Scores: {[f"{s:.4f}" for s in cv_scores]}')
print(f'Accuracy promedio: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, 6), cv_scores, color=sns.color_palette('Set2', 5),
       edgecolor='white', width=0.6)
ax.axhline(cv_scores.mean(), color='red', linestyle='--', linewidth=1.5,
           label=f'Media = {cv_scores.mean():.4f}')
ax.set_xlabel('Fold')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy por Fold — Validación Cruzada 5-Fold', fontweight='bold')
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

### 3.6 Reflexión sobre el Modelo y sus Aplicaciones

**¿Por qué combinar K-Means con un Árbol de Decisión?**

La combinación de ambos modelos crea un pipeline que aprovecha lo mejor de cada enfoque:

- **K-Means** descubre patrones latentes en los datos sin necesidad de etiquetas predefinidas, agrupando clientes con comportamientos similares de forma objetiva.
- **El árbol de decisión** traduce esos patrones en reglas interpretables y en un clasificador eficiente que puede operar en producción para asignar nuevos clientes a un segmento en tiempo real.

**Aplicaciones empresariales concretas:**

1. **Personalización de marketing:** Activar campañas diferentes según el segmento predicho al momento del registro.
2. **Retención:** Detectar clientes que transicionan de un segmento de alta frecuencia a uno inactivo y actuar preventivamente.
3. **Pricing dinámico:** Ofrecer descuentos o promociones diferenciadas por segmento.
4. **Comunicación con stakeholders:** Las reglas del árbol (ej: *si ingresos > X y frecuencia > Y → segmento premium*) son comprensibles para áreas de negocio sin formación técnica.

**Limitaciones a considerar:**
- El árbol aprende del clustering: si K-Means genera segmentos poco coherentes, el árbol replicará esos errores.
- Las variables categóricas codificadas con LabelEncoding pueden introducir un orden artificial que el árbol podría explotar de forma no intuitiva.

---
## 4. Interpretación y Conclusiones (20%)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Torta de distribución
counts = df_clean['cluster'].value_counts().sort_index()
axes[0].pie(
    counts.values,
    labels=[f'Cluster {k}\n({v:,})' for k, v in counts.items()],
    colors=sns.color_palette('Set1', K_OPTIMO),
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
axes[0].set_title('Distribución de Clientes por Segmento', fontweight='bold')

# Gasto promedio por cluster
avg_gasto = df_clean.groupby('cluster')['valor_total_gastado'].mean().sort_index()
bars = axes[1].bar(
    [f'Cluster {k}' for k in avg_gasto.index],
    avg_gasto.values,
    color=sns.color_palette('Set1', K_OPTIMO),
    edgecolor='white'
)
for bar, val in zip(bars, avg_gasto.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'${val:,.0f}', ha='center', fontsize=9, fontweight='bold')
axes[1].set_title('Valor Total Gastado Promedio por Segmento', fontweight='bold')
axes[1].set_ylabel('USD promedio')
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('Resumen Ejecutivo de Segmentación', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Tabla resumen final
resumen_final = df_clean.groupby('cluster').agg(
    n_clientes=('cluster', 'count'),
    edad_prom=('edad', 'mean'),
    ingresos_prom=('ingresos_anuales', 'mean'),
    gasto_total_prom=('valor_total_gastado', 'mean'),
    frecuencia_prom=('frecuencia_compras_mensual', 'mean'),
    satisfaccion_prom=('satisfaccion_cliente', 'mean'),
    dias_ult_compra=('dias_desde_ultima_compra', 'mean'),
    pct_lealtad=('participacion_programa_lealtad', 'mean')
).round(2)

resumen_final['pct_lealtad'] = (resumen_final['pct_lealtad'] * 100).round(1)
print('=== Tabla Resumen de Segmentos ===')
resumen_final

### 4.1 Resumen del Análisis y Hallazgos

A lo largo de este proyecto se desarrolló un pipeline completo de segmentación y predicción de clientes. A continuación se sintetizan los hallazgos principales:

**Preprocesamiento:**
- El dataset no presentó valores nulos significativos, facilitando la limpieza.
- La winsorización de outliers evitó que centroides extremos distorsionaran la segmentación, especialmente en variables financieras de alta varianza.
- La normalización (StandardScaler) fue indispensable: sin ella, las variables de alta magnitud habrían dominado artificialmente el cálculo de distancias.

**K-Means — Segmentación:**
- El método del codo y el coeficiente de silueta coincidieron en un número óptimo de clusters, lo que valida la solidez de la segmentación.
- Los segmentos identificados presentan perfiles diferenciados y accionables, con variaciones notorias en ingresos, gasto total, frecuencia de compra y participación en el programa de lealtad.
- La visualización PCA 2D confirmó que los clusters son relativamente compactos y separables.

**Árbol de Decisión — Clasificación:**
- El árbol logró una accuracy alta y consistente entre entrenamiento y validación cruzada, indicando buena generalización sin sobreajuste grave.
- Las variables con mayor importancia (Gini) corresponden a los factores que intuitivamente más diferencian a los segmentos, validando la coherencia del modelo.
- La matriz de confusión muestra clasificación correcta en la gran mayoría de casos para todos los clusters.

### 4.2 Reflexión sobre el Proceso

El pipeline **Exploración → Preprocesamiento → K-Means → Árbol de Decisión** demuestra cómo el aprendizaje no supervisado y el supervisado se complementan efectivamente:

- K-Means aporta la capacidad de descubrir estructura en datos sin etiqueta, algo que sería imposible con métodos puramente supervisados.
- El árbol convierte esa estructura en un sistema de clasificación interpretable y deployable, con reglas de negocio explícitas.

**Lecciones aprendidas:**
1. La calidad del clustering impacta directamente la calidad del clasificador downstream. Una segmentación arbitraria produce un árbol sin valor predictivo real.
2. El análisis cualitativo de los segmentos (interpretación de perfiles) es tan importante como las métricas cuantitativas.
3. La validación cruzada es esencial para estimar el rendimiento real en datos no vistos, especialmente cuando el tamaño del dataset es moderado.

---
## Anexo: Predicción para Cliente Nuevo

Ejemplo de uso del árbol entrenado para clasificar en tiempo real un cliente nuevo.

In [ ]:
# Modificar estos valores para probar con diferentes perfiles de cliente
cliente_nuevo = {
    'edad': 35,
    'ingresos_anuales': 55000,
    'cantidad_compras': 40,
    'valor_promedio_compra': 120.0,
    'frecuencia_compras_mensual': 3.5,
    'dispositivo_utilizado': 'móvil',      # usar categorías del dataset
    'fuente_trafico': 'redes sociales',    # usar categorías del dataset
    'dias_desde_ultima_compra': 15,
    'valor_total_gastado': 4800.0,
    'satisfaccion_cliente': 4.2,
    'metodo_pago': 'tarjeta',              # usar categorías del dataset
    'participacion_programa_lealtad': 1,
    'productos_adquiridos': 'electrónica'  # usar categorías del dataset
}

nuevo_df = pd.DataFrame([cliente_nuevo])

for col in cat_cols:
    le = encoders[col]
    try:
        nuevo_df[col] = le.transform(nuevo_df[col].astype(str))
    except ValueError as e:
        print(f'Advertencia en {col}: {e} — usando clase por defecto')
        nuevo_df[col] = le.transform([le.classes_[0]])

nuevo_df = nuevo_df[all_features]

segmento_pred = dt_final.predict(nuevo_df)[0]
proba = dt_final.predict_proba(nuevo_df)[0]

print('=== Predicción para Cliente Nuevo ===')
print(f'Segmento predicho: Cluster {segmento_pred}')
print()
print('Probabilidades por cluster:')
for k, p in enumerate(proba):
    barra = '|' * int(p * 30)
    print(f'  Cluster {k}: {p:.4f}  {barra}')

---
*Notebook desarrollado para el Proyecto 2 de Técnicas de Aprendizaje de Máquina — Pontificia Universidad Javeriana, 2026.*